# Quick check: can the arm reach the turned imaging rack?

The imaging rack is now turned a quarter turn: its slots run front to back, and a rod placed in it
lies left to right, with the coated end on the left (towards the camera).

These cells move the **empty** gripper from where it is now (pointing down, above the racks) to
10 cm above one slot of the imaging rack. On the way, high above the bench, the gripper turns a
quarter turn so that it can hold a rod left to right. It can then lower to 5 mm above the slot
(optional), and finally goes back the same way, turning back.

Before running:
- Swing the camera and its boom out of the way.
- Gripper empty, the red wire on the gripper has some slack, hand on the e-stop.
- Do not run anything in another notebook that is connected to the arm at the same time.

Run the cells in order, without restarting the kernel in between.
To try another slot, change `SLOT` at the top of cell 1.

In [ ]:
from robotic_testing.platforms import init_vla_platform

rt = init_vla_platform(exp_name='imaging_quick_check', connect_db=False)
arm = rt.arm

In [ ]:
# ===== Quick check 1/3: EMPTY gripper from where it is now to 10 cm above the TURNED imaging rack =====
# Before running: camera and boom swung out of the way, gripper empty, hand on the e-stop.
# Route: straight up from where the arm is now
#        -> across, high above the bench, turning the gripper a quarter turn on the way
#        -> stops so you can look at the gripper and its red wire
#        -> down to 10 cm above the slot (only after your "yes").
# These moves are not checked against the code's safety zone, so they are slow and need your "yes".

SLOT = 0          # 0 = the slot nearest the rack's top screw (towards the wall);
                  # 1, 2, 3, ... are the next slots towards you, 20 mm apart
SAFE_Z = 500.0    # travel height, well above the flask

# Estimate: the rack was turned about its top screw, which sits where its left screw used to be
# (8 holes = 400 mm left of the rack with sample 0, same distance from the wall).
SLOT_0 = [-251.0, 403.1, 190.6]      # x, y, z of the slot nearest the top screw
RACK_RPY = [180.0, 0.0, 90.0]        # gripper pointing down, rod front-to-back (the normal racks)
IMAGING_RPY = [180.0, 0.0, 0.0]      # gripper pointing down, rod left-to-right, coated end on the left

x, y, z = SLOT_0[0], SLOT_0[1] - 20.0 * SLOT, SLOT_0[2]


def at_speed(mm_s):
    """Normal motion settings, with the speed (mm/s) changed."""
    return {**arm.config.pos_settings_dict['default'], 'speed': mm_s}


start = arm.get_cartesian_pos()          # remembered, so cell 3 can bring the arm back here
print('Arm is now at:', [round(v, 1) for v in start])
turn = max(abs(((a - b) + 180) % 360 - 180) for a, b in zip(start[3:], RACK_RPY))
if turn > 5 or start[2] < 240:
    raise SystemExit('The gripper must point straight down (like above the racks) and be at least '
                     '5 cm above the racks (z >= 240) before this check. The arm did not move.')

steps = [
    ('straight up from here',                  [start[0], start[1], SAFE_Z, *RACK_RPY], 50),
    ('across, turning a quarter turn',         [x, y, SAFE_Z, *IMAGING_RPY], 30),
]
print(f'Plan (slot {SLOT}):')
for name, pose, speed in steps + [('then, after your yes: down to 10 cm above the slot',
                                   [x, y, z + 100, *IMAGING_RPY], 20)]:
    print(f'  {name:52s} -> {[round(v, 1) for v in pose]} at {speed} mm/s')
print('Check the red wire on the gripper now: it needs some slack, because the gripper will turn a quarter turn.')
if input('Camera moved away, gripper empty, red wire has slack, hand on the e-stop? Type yes to start: ').strip().lower() != 'yes':
    raise SystemExit('Cancelled, the arm did not move.')

arm.assert_ready()
for name, pose, speed in steps:
    print('->', name)
    arm.move_to_pos(pose, **at_speed(speed))
print('Now high above the imaging rack at:', [round(v, 1) for v in arm.get_cartesian_pos()])
print('Look at the gripper: the fingers should now be one in front and one behind (towards the wall),')
print('and the red wire should still be loose, not pulled tight.')
if input('All good? Type yes to go down to 10 cm above the slot: ').strip().lower() == 'yes':
    arm.move_to_pos([x, y, z + 100, *IMAGING_RPY], **at_speed(20))
    print('Stopped at:', [round(v, 1) for v in arm.get_cartesian_pos()])
    print(f'Look from the front and from the side: are the fingers right above slot {SLOT} of the turned rack?')
else:
    print('Stayed high in the air. Run cell 3 to bring the arm back.')

In [ ]:
# ===== Quick check 2/3 (optional): lower slowly to 5 mm above the slot =====
target = [x, y, z + 5, *IMAGING_RPY]
here = arm.get_cartesian_pos()
if max(abs(here[0] - x), abs(here[1] - y), abs(here[2] - (z + 100))) > 5:
    print('The arm is not 10 cm above the slot (cell 1 stopped early), so it stays where it is.')
    print('Run cell 3 to bring it back.')
elif input('Fingers right above the slot, nothing in the way? Type yes to go down slowly: ').strip().lower() == 'yes':
    arm.move_to_pos(target, **at_speed(8))
    print('Now at:', [round(v, 1) for v in arm.get_cartesian_pos()])
    print('If the fingers are off the slot, note which way (left / right / towards the wall / towards you)')
    print('and roughly how many mm.')
else:
    print('Stayed where it is.')

In [ ]:
# ===== Quick check 3/3: back to where the arm started (above sample 0) =====
# Route: straight up (no turning) -> across, high above the bench, turning back -> down to the start position.
cur = arm.get_cartesian_pos()
print('Going back: straight up, across (turning the gripper back), down to the start position.')
arm.move_to_pos([cur[0], cur[1], SAFE_Z, *cur[3:]], **at_speed(20))
arm.move_to_pos([start[0], start[1], SAFE_Z, *RACK_RPY], **at_speed(30))
arm.move_to_pos([start[0], start[1], start[2], *RACK_RPY], **at_speed(20))
print('Back at:', [round(v, 1) for v in arm.get_cartesian_pos()])